# DeFoG — QM9 Training Curves (post-hoc evaluation)

Re-evaluates every saved checkpoint of the three QM9 ablation runs with a fixed
sample budget, then plots three publication-quality figures:

1. **VUN** (Validity × Uniqueness × Novelty), linear y-axis
2. **Valency MAE**, log y-axis
3. **Edge MAE**, log y-axis

Each curve covers epochs 19, 39, …, 399 for the three distributions:
`uniform`, `marginal`, and `spminer` (loaded_marginal).

### Why re-evaluate instead of using W&B history
- Mid-training validations used 128 samples (±9% noise on V/U/N → ~15% on VUN).
  Curves are jaggy at that budget.
- The 100-epoch and 100→400-epoch runs were separate W&B runs with different
  log cadences; stitching them visually leaves vertical discontinuities.

Re-evaluating with a uniform 1024-sample budget across all 70+ checkpoints gives
clean, mutually comparable numbers in a single CSV.

### Runtime estimate
~70 checkpoints × ~1.5 min/eval on A100 = **~1.75 hours** for the sweep.
Plot generation: <1 min.

## 1 · Environment setup

Mirrors the QM9 ablation notebook exactly — `colab_create_env.sh` builds the
defog conda env from scratch with all dependency-conflict workarounds baked in.

In [ ]:
from google.colab import drive, userdata
import os

drive.mount('/content/drive')

try:
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print('W&B API key loaded from Colab Secrets.')
except Exception as e:
    print(f'Warning: could not load WANDB_API_KEY ({e}).')

os.environ['MPLBACKEND'] = 'Agg'
print('Drive mounted.')

In [ ]:
!nvidia-smi

In [ ]:
%%bash
set -e
if [ -f /content/miniconda3/bin/conda ]; then
    echo "Conda already installed."
    /content/miniconda3/bin/conda --version
    exit 0
fi
wget -q \
  https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh \
  -O /tmp/miniforge.sh
bash /tmp/miniforge.sh -b -p /content/miniconda3
rm /tmp/miniforge.sh
echo "Miniforge installed."
/content/miniconda3/bin/conda --version

In [ ]:
GITHUB_REPO = 'https://github.com/Maxlyu254/DeFoGPlus.git'
REPO_DIR    = '/content/DeFoGPlus'

print(f'Will clone: {GITHUB_REPO}')

In [ ]:
%%bash -s "$GITHUB_REPO" "$REPO_DIR"
GITHUB_REPO="$1"
REPO_DIR="$2"
set -e
if [ -d "$REPO_DIR/.git" ]; then
    echo "Repo already present — pulling latest."
    git -C "$REPO_DIR" pull
else
    git clone "$GITHUB_REPO" "$REPO_DIR"
fi
echo "Repo ready."
git -C "$REPO_DIR" log --oneline -3

In [ ]:
%%bash
set -e
bash /content/DeFoGPlus/shell/colab_create_env.sh 2>&1

In [ ]:
%%bash -s "$REPO_DIR"
REPO_DIR="$1"
set -e

DRIVE_BASE='/content/drive/MyDrive/DeFoGColab'
mkdir -p "$DRIVE_BASE/checkpoints" "$DRIVE_BASE/data/qm9" "$DRIVE_BASE/figures"

CKPT_LINK="$REPO_DIR/src/checkpoints"
[ -L "$CKPT_LINK" ] && rm "$CKPT_LINK"
ln -s "$DRIVE_BASE/checkpoints" "$CKPT_LINK"
echo "Symlink: $CKPT_LINK -> $DRIVE_BASE/checkpoints"

DATA_LINK="$REPO_DIR/data"
[ -L "$DATA_LINK" ] || [ -d "$DATA_LINK" ] && rm -rf "$DATA_LINK"
ln -s "$DRIVE_BASE/data" "$DATA_LINK"
echo "Symlink: $DATA_LINK -> $DRIVE_BASE/data"

echo "Symlinks ready."

In [ ]:
import subprocess, os

PYTHON      = '/content/miniconda3/envs/defog/bin/python'
DRIVE_BASE  = '/content/drive/MyDrive/DeFoGColab'
LIBGOMP     = '/content/miniconda3/envs/defog/lib/libgomp.so.1'
SRC         = f'{REPO_DIR}/src'
DATA_QM9    = f'{REPO_DIR}/data/qm9'
FIGURES_DIR = f'{DRIVE_BASE}/figures'

BASE_ENV = {
    **os.environ,
    'MPLBACKEND': 'Agg',
    'LD_PRELOAD': LIBGOMP,
    'PYTHONPATH': REPO_DIR,
}

check = '''
import torch, graph_tool, rdkit, torch_geometric, pytorch_lightning
import numpy as np
assert hasattr(np, "Inf")
print("torch          :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU            :", torch.cuda.get_device_name(0))
'''
r = subprocess.run([PYTHON, '-c', check], capture_output=True, text=True, env=BASE_ENV)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr)
    raise RuntimeError('Import check failed.')
print('Environment ready.')

## 2 · Plot configuration

**Edit this cell to adjust titles, axis labels, curve names, and styling.**
All visual customization for the figures lives here.

In [ ]:
# ── Run identifiers (used for both evaluation and plotting) ──────────────────
#   key    : short symbolic name used internally
#   ckpt_dir_name : actual subdirectory name under src/checkpoints/
#   display_name  : how the curve is labelled in figure legends
RUNS = [
    {'key': 'uniform',  'ckpt_dir_name': 'qm9_ablation_uniform',  'display_name': 'Uniform'},
    {'key': 'marginal', 'ckpt_dir_name': 'qm9_ablation_marginal', 'display_name': 'Marginal'},
    {'key': 'spminer',  'ckpt_dir_name': 'qm9_ablation_spminer',  'display_name': 'SPMiner motif-edited marginal'},
]

# ── Per-run visual style ──────────────────────────────────────────────────────
# CB-friendly palette + distinct linestyle/marker so curves are readable in B/W.
STYLE = {
    'uniform':  {'color': '#1f3a93', 'linestyle': '-',  'marker': 'o'},  # navy
    'marginal': {'color': '#d35400', 'linestyle': '--', 'marker': 's'},  # dark orange
    'spminer':  {'color': '#117a3a', 'linestyle': ':',  'marker': '^'},  # dark green
}

# ── Per-figure configuration ──────────────────────────────────────────────────
# Three figures, one per metric.
FIGURES = [
    {
        'metric_column': 'vun',
        'filename':      'qm9_curves_vun.png',
        'title':         'VUN over training (QM9)',
        'ylabel':        r'V $\times$ U $\times$ N',
        'yscale':        'linear',
        'ylim':          (0.0, 1.0),
    },
    {
        'metric_column': 'valency_mae',
        'filename':      'qm9_curves_valency_mae.png',
        'title':         'Valency MAE over training (QM9)',
        'ylabel':        'Valency MAE',
        'yscale':        'log',
        'ylim':          None,
    },
    {
        'metric_column': 'edge_mae',
        'filename':      'qm9_curves_edge_mae.png',
        'title':         'Edge MAE over training (QM9)',
        'ylabel':        'Edge MAE',
        'yscale':        'log',
        'ylim':          None,
    },
]

# ── Shared axis / typographic settings ────────────────────────────────────────
X_LABEL          = 'Epoch'
X_LIM            = (0, 400)
FIGSIZE          = (6.4, 4.0)        # inches; single-column paper width
DPI              = 300
FONT_FAMILY      = 'serif'
FONT_SIZE_BODY   = 11
FONT_SIZE_AXIS   = 12
FONT_SIZE_LEGEND = 10
FONT_SIZE_TITLE  = 12
LINEWIDTH        = 1.6
MARKERSIZE       = 5
LEGEND_LOC       = 'best'             # matplotlib legend location
GRID_KW          = {'linestyle': '--', 'linewidth': 0.5, 'alpha': 0.6}

# ── Sweep settings ────────────────────────────────────────────────────────────
EPOCHS_TO_EVAL    = list(range(19, 400, 20))   # [19, 39, ..., 399]  — 20 points
N_SAMPLES         = 1024                       # samples per checkpoint
SAMPLE_STEPS      = 500                        # paper default for QM9
CSV_PATH          = f'{DATA_QM9}/curves.csv'   # persisted on Drive via symlink

print(f'Configured {len(RUNS)} runs × {len(EPOCHS_TO_EVAL)} epochs = '
      f'{len(RUNS) * len(EPOCHS_TO_EVAL)} evaluations total.')
print(f'Sample budget: {N_SAMPLES} per checkpoint × sample_steps={SAMPLE_STEPS}.')
print(f'CSV output: {CSV_PATH}')
print(f'Figures output: {FIGURES_DIR}/')

## 3 · Helpers

Checkpoint discovery, output parsing, evaluation runner. Sweep is
**resume-aware**: skips evaluations already present in the CSV.

In [ ]:
import re
import csv
import glob
from pathlib import Path


# Regex patterns for parsing DeFoG's stdout. Lines printed by:
#   src/analysis/rdkit_functions.py  -> Validity / Uniqueness / Novelty
#   src/metrics/molecular_metrics.py -> BasicMetricsMAE
RE_VALIDITY    = re.compile(r'Validity over \d+ molecules:\s*([\d.]+)%')
RE_UNIQUENESS  = re.compile(r'Uniqueness over \d+ valid molecules:\s*([\d.]+)%')
RE_NOVELTY     = re.compile(r'Novelty over \d+ unique valid molecules:\s*([\d.]+)%')
RE_BASIC_MAE   = re.compile(
    r'BasicMetricsMAE:\s*'
    r'n_mae=([\d.eE+-]+)\s+'
    r'node_mae=([\d.eE+-]+)\s+'
    r'edge_mae=([\d.eE+-]+)\s+'
    r'valency_mae=([\d.eE+-]+)'
)

# Lightning's checkpoint filename is typically `epoch=N-step=M.ckpt`.
RE_EPOCH_FROM_FILENAME = re.compile(r'epoch=(\d+)')


def list_checkpoints(ckpt_dir_name):
    """Return list of (epoch:int, ckpt_path:str) for one run, sorted by epoch."""
    ckpt_dir = f'{SRC}/checkpoints/{ckpt_dir_name}'
    out = []
    for path in glob.glob(f'{ckpt_dir}/*.ckpt'):
        m = RE_EPOCH_FROM_FILENAME.search(os.path.basename(path))
        if m:
            out.append((int(m.group(1)), path))
    out.sort()
    return out


def find_checkpoint_at_epoch(ckpt_dir_name, target_epoch):
    """Return the checkpoint path whose filename epoch matches target_epoch, or None."""
    for ep, path in list_checkpoints(ckpt_dir_name):
        if ep == target_epoch:
            return path
    return None


def parse_metrics_from_output(stdout):
    """Parse V, U, N, *_MAE values from a test run's stdout. Returns dict.

    Multiple matches can appear in a long log; we take the LAST occurrence so
    that mid-validation prints (if any) don't shadow the final test number.
    Missing values become None.
    """
    def _last(pattern, text, group_count=1):
        matches = list(pattern.finditer(text))
        if not matches:
            return None if group_count == 1 else (None,) * group_count
        m = matches[-1]
        if group_count == 1:
            return float(m.group(1))
        return tuple(float(m.group(i)) for i in range(1, group_count + 1))

    validity   = _last(RE_VALIDITY, stdout)
    uniqueness = _last(RE_UNIQUENESS, stdout)
    novelty    = _last(RE_NOVELTY, stdout)
    n_mae, node_mae, edge_mae, valency_mae = _last(RE_BASIC_MAE, stdout, 4)

    # Validity/uniqueness/novelty in stdout are percentages; normalize to [0,1].
    if validity is not None:   validity   /= 100.0
    if uniqueness is not None: uniqueness /= 100.0
    if novelty is not None:    novelty    /= 100.0

    # VUN = product of the three. If any component is missing/zero, propagate NaN.
    if validity is None or uniqueness is None or novelty is None:
        vun = None
    else:
        vun = validity * uniqueness * novelty

    return {
        'validity':    validity,
        'uniqueness':  uniqueness,
        'novelty':     novelty,
        'vun':         vun,
        'n_mae':       n_mae,
        'node_mae':    node_mae,
        'edge_mae':    edge_mae,
        'valency_mae': valency_mae,
    }


def evaluate_checkpoint(ckpt_dir_name, ckpt_path, n_samples, sample_steps):
    """Run main.py with test_only=<ckpt> and parse metrics. Returns dict.

    Sets general.wandb=disabled so we don't litter the W&B project with one
    run per checkpoint.
    """
    # Pick the experiment-level extras based on the run's transition.
    # We don't actually need them for test-only (the model architecture is
    # restored from the ckpt), but DeFoG's main.py validates the config.
    # Use neutral defaults that won't interfere with sampling.
    args = [
        '+experiment=qm9_no_h',
        'hydra.job.chdir=False',
        'general.wandb=disabled',
        f'general.name={ckpt_dir_name}',
        f'general.final_model_samples_to_generate={n_samples}',
        f'sample.sample_steps={sample_steps}',
        # Single-quote: ckpt filenames contain '=' which breaks Hydra's parser.
        f"general.test_only='{ckpt_path}'",
    ]

    # Don't print the entire stdout to the cell — capture it and parse.
    proc = subprocess.run(
        [PYTHON, 'main.py'] + args,
        cwd=SRC, env=BASE_ENV,
        capture_output=True, text=True,
    )
    if proc.returncode != 0:
        # Surface stderr tail for debugging; the full stdout is large.
        tail = '\n'.join(proc.stdout.splitlines()[-40:])
        raise RuntimeError(
            f'Test run failed (exit {proc.returncode}). Last stdout lines:\n{tail}\n'
            f'Stderr tail:\n{proc.stderr[-2000:]}'
        )
    return parse_metrics_from_output(proc.stdout)


def load_existing_csv(csv_path):
    """Return set of (run_key, epoch) already present in the CSV."""
    done = set()
    if not os.path.isfile(csv_path):
        return done
    with open(csv_path, 'r', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            done.add((row['run'], int(row['epoch'])))
    return done


CSV_FIELDS = [
    'run', 'epoch', 'n_samples', 'sample_steps',
    'validity', 'uniqueness', 'novelty', 'vun',
    'n_mae', 'node_mae', 'edge_mae', 'valency_mae',
]


def append_csv_row(csv_path, row):
    """Append one row, creating the file with a header if it doesn't exist."""
    new_file = not os.path.isfile(csv_path)
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    with open(csv_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        if new_file:
            writer.writeheader()
        writer.writerow(row)


print('Helpers loaded.')

## 4 · Pre-flight: list checkpoints to evaluate

Show what's missing/present so you can confirm before running the sweep.

In [ ]:
done = load_existing_csv(CSV_PATH)
print(f'CSV: {CSV_PATH}')
print(f'Already evaluated: {len(done)} (run, epoch) pairs.\n')

todo = []
missing = []
for run in RUNS:
    print(f'{run["display_name"]:>32s}  ({run["ckpt_dir_name"]})')
    have_epochs = {ep for ep, _ in list_checkpoints(run['ckpt_dir_name'])}
    for epoch in EPOCHS_TO_EVAL:
        ckpt = find_checkpoint_at_epoch(run['ckpt_dir_name'], epoch)
        key = (run['key'], epoch)
        if key in done:
            status = '✓ done'
        elif ckpt is None:
            status = '✗ missing checkpoint'
            missing.append((run['key'], epoch))
        else:
            status = '… queued'
            todo.append((run, epoch, ckpt))
        print(f'  epoch {epoch:>3}: {status}')
    print()

print(f'Summary: {len(todo)} to evaluate, {len(done)} already done, {len(missing)} missing checkpoints.')
if todo:
    est_min = len(todo) * 1.5  # ~1.5 min/eval at 1024 samples on A100
    print(f'Estimated sweep time: ~{est_min:.0f} min ({est_min/60:.1f} h).')

## 5 · Sweep: re-evaluate every checkpoint

Each evaluation is appended to the CSV immediately, so the sweep is
interruption-safe. Re-running the cell after a session timeout continues
from where it stopped.

In [ ]:
import time

for i, (run, epoch, ckpt) in enumerate(todo, 1):
    t0 = time.time()
    print(f'[{i}/{len(todo)}] {run["display_name"]} @ epoch {epoch}')
    print(f'        ckpt: {os.path.basename(ckpt)}')

    metrics = evaluate_checkpoint(run['ckpt_dir_name'], ckpt, N_SAMPLES, SAMPLE_STEPS)

    row = {
        'run':          run['key'],
        'epoch':        epoch,
        'n_samples':    N_SAMPLES,
        'sample_steps': SAMPLE_STEPS,
        **metrics,
    }
    append_csv_row(CSV_PATH, row)

    dt = time.time() - t0
    print(f'        V={metrics["validity"]}  U={metrics["uniqueness"]}  '
          f'N={metrics["novelty"]}  VUN={metrics["vun"]}')
    print(f'        valency_mae={metrics["valency_mae"]}  '
          f'edge_mae={metrics["edge_mae"]}')
    print(f'        elapsed: {dt:.0f}s')
    print()

print(f'Sweep complete. CSV at {CSV_PATH}.')

## 6 · Generate figures

Reads the CSV and writes 3 PNGs to `MyDrive/DeFoGColab/figures/`.

Re-runnable independently from the sweep — adjust styling in the
**plot configuration** cell above and re-run this cell to regenerate.

In [ ]:
# Plot generation runs in the *current* notebook kernel (Colab's system Python),
# not in the conda env. Colab ships matplotlib + pandas already.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv(CSV_PATH)
print(f'Loaded {len(df)} rows from {CSV_PATH}')
print(df.groupby('run').size())

os.makedirs(FIGURES_DIR, exist_ok=True)

# Apply global style.
plt.rcParams.update({
    'font.family':       FONT_FAMILY,
    'font.size':         FONT_SIZE_BODY,
    'axes.labelsize':    FONT_SIZE_AXIS,
    'axes.titlesize':    FONT_SIZE_TITLE,
    'legend.fontsize':   FONT_SIZE_LEGEND,
    'xtick.labelsize':   FONT_SIZE_BODY,
    'ytick.labelsize':   FONT_SIZE_BODY,
    'axes.spines.top':   True,
    'axes.spines.right': True,
})

for fig_cfg in FIGURES:
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

    for run in RUNS:
        sub = df[df['run'] == run['key']].sort_values('epoch')
        if sub.empty:
            print(f'  warning: no data for run {run["key"]!r}')
            continue
        # Drop any rows where the metric is missing.
        sub = sub.dropna(subset=[fig_cfg['metric_column']])
        if sub.empty:
            print(f'  warning: all rows for {run["key"]!r} have missing '
                  f'{fig_cfg["metric_column"]}')
            continue

        style = STYLE[run['key']]
        ax.plot(
            sub['epoch'], sub[fig_cfg['metric_column']],
            label=run['display_name'],
            color=style['color'],
            linestyle=style['linestyle'],
            marker=style['marker'],
            linewidth=LINEWIDTH,
            markersize=MARKERSIZE,
        )

    ax.set_xlabel(X_LABEL)
    ax.set_ylabel(fig_cfg['ylabel'])
    ax.set_title(fig_cfg['title'])
    ax.set_xlim(*X_LIM)
    ax.set_yscale(fig_cfg['yscale'])
    if fig_cfg['ylim'] is not None:
        ax.set_ylim(*fig_cfg['ylim'])
    ax.grid(True, **GRID_KW)
    ax.legend(loc=LEGEND_LOC, frameon=True, framealpha=0.9)

    out_path = f'{FIGURES_DIR}/{fig_cfg["filename"]}'
    fig.tight_layout()
    fig.savefig(out_path, dpi=DPI, bbox_inches='tight')
    plt.close(fig)
    print(f'  saved → {out_path}')

print('\nAll figures saved.')
print('Download from Drive: MyDrive/DeFoGColab/figures/')